### 뉴스 데이터 분석 자동화

### 뉴스데이터가 1000개
- 본문을 읽어서 정형 데이터로 변환하는 자동화 파이프라인을 만들어 봅시다
- 1000개 데이터의 내용을 카테고리는? 핵심어는? 요약하면? 의 세가지 키워드로 파악
- 항목
    - 카테고리
    - 요약
    - 핵심어
- 구조화된 출력으로!

### 파이프라인 설계
1. 읽기 : 뉴스 본문 불러오기
2. 분석 : 각 기사를 정해진 항목으로 분석
3. 정리, 저장 : 표로 모아서 csv 저장

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [7]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv")
df = df.head(5)

In [8]:
# 1. 일단 tool을 만들자

def check_data(column):  # 본문 물어보면 본문, 제목 물어보면 제목 도출
    df = pd.read_csv("../data/11-1_뉴스정제.csv")
    df = df.head(5)
    return df[column].to_json(force_ascii=False)

def save(data):
    with open ("result.csv", "w", encoding="utf-8-sig") as file:
        file.write(data)

    return "저장 완료"


In [9]:
tools = [
      {
          "type": "function",
          "function": {
              "name": "check_data",
              "description": "뉴스 데이터에서 원하는 열을 불러온다.",
              "parameters": {
                  "type": "object",
                  "properties": {
                      "column": {
                          "type": "string",
                          "description": "불러올 열 이름"
                      }
                  },
                  "required": ["column"]
              }
          }
      },
      {
          "type": "function",
          "function": {
              "name": "save",
              "description": "분석 결과를 CSV 파일로 저장한다.",
              "parameters": {
                  "type": "object",
                  "properties": {
                      "data": {
                          "type": "string",
                          "description": "저장할 CSV 형식 문자열"
                      }
                  },
                  "required": ["data"]
              }
          }
      }
  ]

In [ ]:
# # system prompt 작성

# system_prompt = """
# 당신은 뉴스 데이터 분석 파이프라인의 분석 담당자입니다.

# 목표는 뉴스 기사마다 아래 항목을 추출하여 저장하는 것입니다.

# [분석 항목]
# - category: 정치, 경제, 사회, 국제, 문화, 스포츠, IT/과학 중 하나
# - summary: 기사의 핵심 내용을 1~2문장으로 요약
# - keywords: 핵심어 3개를 쉼표로 구분

# 작업 절차:
# 1. 먼저 check_data 도구를 사용하여 뉴스 본문 데이터를 불러옵니다.
# 2. 기사별로 category, summary, keywords를 분석합니다.
# 3. 원문에 없는 사실은 추가하지 않습니다.
# 4. 카테고리는 반드시 지정된 목록 중 하나만 사용합니다.
# 5. 요약은 의견이나 평가 없이 기사에 나온 사실 중심으로 작성합니다.
# 6. 핵심어는 중복 없이 중요한 단어 3개를 작성합니다.
# 7. 모든 기사의 분석이 끝난 뒤, 결과를 아래 열 순서의 CSV 텍스트로 만듭니다.

# category,summary,keywords

# 8. 완성된 CSV 텍스트 전체를 save 도구의 data 인자로 전달합니다.
# 9. 저장 전에는 save 도구를 호출하지 않습니다.
# 10. 저장이 끝나면 “저장이 완료되었습니다.”라고만 답합니다.
# """

# """
# 당신은 뉴스 기사 분석가입니다.

#   전달받은 뉴스 기사 본문 1개를 분석하세요.

#   [분석 항목]
#   - category: 정치, 경제, 사회, 국제, 문화, 스포츠, IT/과학 중 하나
#   - summary: 핵심 내용을 객관적으로 1~2문장으로 요약
#   - keywords: 중요한 핵심어 3개

#   [규칙]
#   1. 본문에 없는 사실, 숫자, 인물, 원인을 추측해 추가하지 마세요.
#   2. category는 지정된 목록 중 하나만 선택하세요.
#   3. summary에는 개인 의견이나 평가를 넣지 마세요.
#   4. keywords는 중복 없이 작성하세요.
#   5. 기사 제목이나 본문에 없는 내용을 만들지 마세요.
#   6. 아래 JSON 형식만 출력하세요. Markdown 코드 블록은 사용하지 마세요.

#   {
#     "category": "카테고리",
#     "summary": "요약",
#     "keywords": ["핵심어1", "핵심어2", "핵심어3"]
#   }
# """

# """
# 당신은 뉴스 분석 및 저장 파이프라인을 수행하는 에이전트입니다.

#   반드시 다음 순서로 작업하세요.

#   1. check_data 도구를 호출하여 "본문" 열의 뉴스 5개를 가져옵니다.
#   2. 각 기사마다 category, summary, keywords를 분석합니다.
#   3. 분석 결과 전체를 하나의 CSV 텍스트로 만듭니다.
#   4. CSV 열 순서는 article_id, category, summary, keywords입니다.
#   5. keywords는 쉼표가 아닌 | 기호로 구분합니다.
#   6. 분석 결과를 만들기 전에는 save 도구를 호출하지 않습니다.
#   7. 모든 기사 분석이 끝난 뒤 save 도구를 정확히 한 번 호출합니다.
#   8. save의 data 인자에는 설명 없이 CSV 텍스트만 전달합니다.
#   9. 저장 완료 도구 결과를 받은 뒤, 사용자에게 저장 완료 사실만 짧게 답합니다.

#   분석 규칙:
#   - category는 정치, 경제, 사회, 국제, 문화, 스포츠, IT/과학 중 하나입니다.
#   - 본문에 없는 사실을 추가하지 않습니다.
#   - summary는 객관적인 1~2문장입니다.
#   - keywords는 중복 없는 핵심어 3개입니다.
# """

In [10]:
system_prompt = """
  당신은 뉴스 분석 자동화 도우미입니다.

  1. check_data 도구로 본문 열을 불러오세요.
  2. 기사 5개를 각각 분석하세요.
  3. 각 기사에서 category, summary, keywords를 작성하세요.
  4. 분석 결과를 아래 열 순서의 CSV 형식으로 만드세요.

  category,summary,keywords

  5. 완성한 CSV 전체를 save 도구로 한 번 저장하세요.
  6. 저장이 끝나면 사용자에게 저장 완료라고 답하세요.

  규칙:
  - category는 정치, 경제, 사회, 국제, 문화, 스포츠, IT/과학 중 하나입니다.
  - summary는 기사 내용 중심으로 짧게 작성합니다.
  - keywords는 핵심어 3개를 | 기호로 구분합니다.
  """

In [11]:
import json
available_tools = {"check_data": check_data, "save" : save}

# def chat_with_tools(question, tools):
#     messages = [{"role": "user", "content": question}]
#     r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
#                                        reasoning_effort="none", messages=messages)
#     calls = r.choices[0].message.tool_calls
#     if not calls:                       # 도구가 필요 없으면 바로 답
#         return r.choices[0].message.content
#     messages.append(r.choices[0].message)
#     for tc in calls:                    # 필요한 도구를 모두 실행
#         args = json.loads(tc.function.arguments)
#         result = available_tools[tc.function.name](**args)
#         messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
#     r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
#                                         reasoning_effort="none", messages=messages)
#     return r2.choices[0].message.content

def chat_with_tools(question, tools):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    while True:
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )

        message = response.choices[0].message
        tool_calls = message.tool_calls

        # 더 이상 실행할 도구가 없으면 최종 답변 반환
        if not tool_calls:
            return message.content

        messages.append(message)

        for tool_call in tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            result = available_tools[name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })


In [12]:
result = chat_with_tools(
    "뉴스 본문 5개를 분석해서 CSV로 저장해줘.",
    tools
)

print(result)

저장 완료했습니다.
